# AsyncUA-Fabrikclients: Publisher & Subscriber fuer `M01.Temperature`

Dieses Notebook ist dafuer ausgelegt, **mit dem bestehenden asynchronen Fabrik-OPC-UA-Server** aus
`01_OPC_Server_template_de.ipynb` zusammenzuarbeiten.

Es implementiert zwei separate Clients:

1. **Publisher-Client**
   - verbindet sich mit dem laufenden Server
   - sucht das Objekt `M01`
   - schreibt regelmaessig neue Werte in die Variable `Temperature`

2. **Subscriber-Client**
   - verbindet sich mit demselben Server
   - abonniert `M01.Temperature`
   - reagiert auf Datenaenderungen ueber einen Callback

> **Voraussetzung:** Der Server aus `01_OPC_Server_template_de.ipynb` muss bereits laufen
> (d. h. `FactoryOpcUaServer().run()` ist aktiv).


## 1. Benötigte Pakete installieren

Falls `asyncua`, `nest_asyncio` und `wait-for2==0.3.2` noch nicht installiert sind, führen Sie die folgende Zelle einmalig aus.


In [ ]:
#!pip install -U asyncua nest_asyncio wait-for2==0.3.2

## 2. Imports und Event-Loop-Einrichtung für Jupyter

Jupyter betreibt bereits eine Event-Loop. Mit `nest_asyncio` können wir `await` in Notebook-Zellen
trotzdem bequem nutzen, ohne in Loop-Konflikte zu geraten.


In [ ]:
import asyncio
import math
from datetime import datetime

import nest_asyncio
nest_asyncio.apply()

from asyncua import ua, Client

# Konfiguration – diese Werte müssen mit dem Server-Notebook übereinstimmen
SERVER_URL = "opc.tcp://localhost:4840/freeopcua/server/"
FACTORY_NS_URI = "http://ostfalia.de/ipt/factory"

print("Event-Loop eingerichtet. Server-Endpunkt:", SERVER_URL)

## 3. Hilfsfunktion: Den Knoten `M01.Temperature` ermitteln

Wir gehen davon aus, dass der Server die folgende Struktur angelegt hat (wie im Template-Notebook):

- `Objects`
  - `Factory`
    - `Machines`
      - `M01`
        - `Temperature`

Die folgende Hilfsfunktion kapselt das Auflösen dieses Browse-Pfads,
sodass beide Clients sie wiederverwenden können.


In [ ]:
async def get_machine_temperature_node(client: Client, machine: str = "M01"):
    """Gibt den Knoten für Factory.Machines.M01.Temperature zurück."""
    # Namespace-Index anhand der URI bestimmen
    nsidx = await client.get_namespace_index(FACTORY_NS_URI)
    print(f"[CLIENT] Namespace index for {FACTORY_NS_URI!r}: {nsidx}")

    objects = client.nodes.objects
    factory = await objects.get_child([f"{nsidx}:Factory"])           # Objects/Factory
    machines_folder = await factory.get_child([f"{nsidx}:Machines"])  # Factory/Machines
    mXY = await machines_folder.get_child([f"{nsidx}:{machine}"])     # Machines/"Maschine"
    temp_node = await mXY.get_child([f"{nsidx}:Temperature"])         # "Maschine"/Temperature

    print(f"[CLIENT] Resolved temperature NodeId: {temp_node.nodeid}")
    return temp_node

## 4. Publisher-Client

Der Publisher-Client verbindet sich mit dem Server und schreibt regelmäßig neue Werte
in `M01.Temperature`. Hier verwenden wir eine einfache Sinuswelle auf einer Basistemperatur.


In [ ]:
async def publisher_client(runtime_seconds: float = 20.0, interval: float = 1.0):
    """Schreibt regelmäßig neue Temperaturwerte in M01.Temperature."""
    async with Client(url=SERVER_URL) as client:
        print("[PUBLISHER] Verbunden mit Server:", SERVER_URL)
        temp_node = await get_machine_temperature_node(client, "M01")

        loop = asyncio.get_running_loop()
        start = loop.time()
        t = 0.0
        while True:
            now = loop.time()
            if now - start > runtime_seconds:
                break

            # Beispiel: Temperatur als (25 + 5 * sin(t))
            value = 25.0 + 5.0 * math.sin(t)
            await temp_node.write_value(ua.Variant(value, ua.VariantType.Double))
            ts = datetime.now().strftime("%H:%M:%S")
            print(f"[PUBLISHER {ts}] Geschriebene Temperatur: {value:.2f} °C")

            t += interval
            await asyncio.sleep(interval)

        print("[PUBLISHER] Abgeschlossen.")

## 5. Subscriber-Client

Der Subscriber-Client:

- verbindet sich mit dem Server,
- löst `M01.Temperature` auf,
- erstellt ein Abonnement (Subscription),
- registriert einen Handler, der bei jeder Wertänderung aufgerufen wird.

### Warum hat der Subscriber ein `publishing_interval_ms`?

In OPC UA besitzt eine **Subscription** ein **Publishing Interval**. Dies ist **nicht** das Intervall,
in dem die serverseitige Logik den Wert aktualisiert, und auch nicht das Intervall des Publisher-Clients.

Es teilt dem Server stattdessen mit:

> *„Bitte prüfe auf Änderungen und sende Publish-Antworten ungefähr alle X Millisekunden.“*

Das bedeutet:

- Der **Publisher-Client** steuert, wie oft Werte geschrieben werden (z. B. jede Sekunde).
- Das `publishing_interval_ms` der **Subscription** steuert, wie oft der Client Updates erwartet
  (z. B. alle 500 ms).

Diese beiden Raten können abweichen – OPC UA übernimmt das Puffern und Zusammenführen der Updates.


In [ ]:
class TemperatureSubHandler:
    """Callback-Handler, der auf Datenaenderungen und Ereignisse reagiert."""

    def datachange_notification(self, node, val, data):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[SUBSCRIBER {ts}] DataChange: Node={node}, Value={val}")

    def event_notification(self, event):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[SUBSCRIBER {ts}] Event: {event}")

async def subscriber_client(runtime_seconds: float = 30.0, publishing_interval_ms: int = 500):
    """Subscribe to M01.Temperature and print all changes.

    :param runtime_seconds: How long the subscriber should stay active.
    :param publishing_interval_ms: OPC UA subscription publishing interval in milliseconds.
                                   This defines how often the server sends publish responses,
                                   not how often the value is written by the publisher client.
    """
    handler = TemperatureSubHandler()

    async with Client(url=SERVER_URL) as client:
        print("[SUBSCRIBER] Verbunden mit Server:", SERVER_URL)
        temp_node = await get_machine_temperature_node(client, "M01")

        # create_subscription gibt ein Subscription-Objekt in asyncua zurueck
        subscription = await client.create_subscription(publishing_interval_ms, handler)

        # subscribe_data_change gibt einen Handle zurueck – nuetzlich, um spaeter einzelne Knoten abzumelden
        handle = await subscription.subscribe_data_change(temp_node)
        print(f"[SUBSCRIBER] Subscription aktiv (handle={handle}) fuer ca. {runtime_seconds} Sekunden ...")

        try:
            await asyncio.sleep(runtime_seconds)
        finally:
            print("[SUBSCRIBER] Subscription wird geloescht ...")
            await subscription.delete()
            print("[SUBSCRIBER] Abgeschlossen.")

## 6. Demo: Publisher und Subscriber parallel ausführen

Die folgende Funktion startet beide Clients gleichzeitig:

- der **Publisher** schreibt für eine begrenzte Zeit,
- der **Subscriber** bleibt etwas länger aktiv,
- beide werden gemeinsam abgewartet.

> **Wichtig:** Stelle sicher, dass dein Server-Notebook bereits läuft, bevor du diese Zelle ausführst.


In [ ]:
async def run_pub_sub_demo():
    """Publisher und Subscriber gleichzeitig gegen den laufenden Fabrik-Server ausführen."""
    pub_task = asyncio.create_task(
        publisher_client(runtime_seconds=20.0, interval=1.0)
    )
    sub_task = asyncio.create_task(
        subscriber_client(runtime_seconds=30.0, publishing_interval_ms=500)
    )

    try:
        await asyncio.gather(pub_task, sub_task)
    except asyncio.CancelledError:
        print("[MAIN] Aufgaben wurden abgebrochen.")


# Demo direkt aus dem Notebook ausführen
await run_pub_sub_demo()

## 7. Mögliche Erweiterungen

- Mehrere Maschinen (`M01`–`M05`) parallel steuern und jeweils abonnieren.
- Weitere Variablen pro Maschine (z. B. `State`, `Busy`) hinzufügen und in die Clients aufnehmen.
